# Workshop: Flow Matching y Diffusion Models

**Autor:** Luis Leal.

### *De ruido a datos: trayectorias en espacios de alta dimensión*

Este taller cubre los **modelos generativos basados en transporte continuo**: Flow Matching y Diffusion Models. La idea central, en una frase:

> Cada muestra (una imagen, un punto 2D, lo que sea) es una **partícula** que vive en un espacio de muchas dimensiones. El modelo aprende un **campo de velocidades** que mueve estas partículas desde una distribución simple (ruido Gaussiano) hasta la distribución de los datos.

Es la matemática de la **mecánica de fluidos**: la densidad de probabilidad evoluciona en el tiempo siguiendo una **ecuación de continuidad**, y las partículas son arrastradas por el flujo. Si encima les damos "patadas" aleatorias (movimiento Browniano), tenemos **diffusion**.

| # | Sección | Tiempo |
|---|---------|--------|
| 0 | Motivación: modelos famosos, datos como vectores, generación | 15 min |
| 1 | Modelado generativo como muestreo | 10 min |
| 2 | Sampling con un modelo pre-entrenado | 15 min |
| 3 | **Refresher: ODEs y campos vectoriales** (con animaciones) | 15 min |
| 4 | Probability paths y vector fields | 10 min |
| 5 | **Flow Matching** en 2D (con trayectorias animadas) | 20 min |
| 6 | Score function y SDEs | 10 min |
| 7 | **Diffusion** en 2D + comparación con FM | 15 min |
| 8 | Arquitecturas: U-Net y time embeddings | 10 min |
| 9 | **Diffusion sobre MNIST** (con denoising animado) | 25 min |
| 10 | Classifier-Free Guidance | 15 min |
| 11 | Latent Diffusion (concepto + demo) | 10 min |

---
## Setup del entorno

Ejecuta las siguientes celdas para instalar las dependencias. En Colab esto toma ~1-2 minutos.  
**IMPORTANTE: Activa GPU**: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`.


In [ ]:
# Dependencias (en Colab)
!pip install -q diffusers transformers accelerate torchvision scikit-learn


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib import cm
from IPython.display import HTML, display
import math

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")
torch.manual_seed(0); np.random.seed(0)

# Reducimos el tamaño de animaciones inline para que el notebook no pese demasiado
plt.rcParams['animation.embed_limit'] = 50  # MB


---
# 0. Motivación y contexto

## 0.1 ¿Por qué importa? La revolución Diffusion & Flow Matching

Desde 2022, prácticamente **todos** los avances de frontera en generación de imágenes, video, audio y hasta predicción de estructuras de proteínas usan alguna variante de **Diffusion Models** o **Flow Matching**:

- **Imágenes:** Stable Diffusion, DALL·E 3, Midjourney, Flux, Imagen 3
- **Video:** Sora (OpenAI), Veo 3 (Google), Runway Gen-3
- **Audio:** Stable Audio, Seed-Music
- **Ciencia:** AlphaFold 3 usa difusión para predecir estructuras moleculares

**Flow Matching** en particular está en plena explosión: de ser un marco teórico nuevo en 2023 a tener **+150 artículos** enviados a ICLR 2026. Modelos como **Flux** (Black Forest Labs) ya lo usan como base.

A continuación, una galería de modelos reales que usan estas técnicas 👇

In [ ]:
# Galería de modelos famosos basados en Diffusion / Flow Matching
from IPython.display import HTML

gallery_html = """
<style>
.model-gallery {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(280px, 1fr));
    gap: 16px;
    padding: 10px 0;
    font-family: system-ui, -apple-system, sans-serif;
}
.model-card {
    border: 1px solid #e0e0e0;
    border-radius: 12px;
    overflow: hidden;
    background: #fafafa;
    transition: box-shadow 0.2s;
}
.model-card:hover {
    box-shadow: 0 4px 12px rgba(0,0,0,0.15);
}
.model-card img {
    width: 100%;
    height: 180px;
    object-fit: cover;
    display: block;
    background: #e8e8e8;
}
.model-card .placeholder {
    width: 100%;
    height: 180px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 64px;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
}
.model-card .info {
    padding: 12px 14px;
}
.model-card .info h3 {
    margin: 0 0 4px 0;
    font-size: 16px;
    color: #1a1a1a;
}
.model-card .info .company {
    color: #666;
    font-size: 13px;
    margin: 0 0 6px 0;
}
.model-card .info .tags {
    display: flex;
    gap: 6px;
    flex-wrap: wrap;
}
.model-card .info .tag {
    font-size: 11px;
    padding: 2px 8px;
    border-radius: 10px;
    font-weight: 600;
}
.tag-diffusion { background: #dbeafe; color: #1e40af; }
.tag-fm { background: #d1fae5; color: #065f46; }
.tag-imagen { background: #fef3c7; color: #92400e; }
.tag-video { background: #fce7f3; color: #9d174d; }
.tag-ciencia { background: #e0e7ff; color: #3730a3; }
.tag-audio { background: #f3e8ff; color: #6b21a8; }
</style>

<div class="model-gallery">

    <div class="model-card">
        <img src="https://huggingface.co/stabilityai/stable-diffusion-3.5-large/media/main/sd3.5_large_demo.png"
             alt="Stable Diffusion 3.5"
             onerror="this.outerHTML='<div class=placeholder>🖼️</div>'">
        <div class="info">
            <h3>Stable Diffusion 3.5</h3>
            <p class="company">Stability AI · 2024</p>
            <div class="tags">
                <span class="tag tag-diffusion">Diffusion (MMDiT)</span>
                <span class="tag tag-imagen">Imagen</span>
            </div>
        </div>
    </div>

    <div class="model-card">
        <img src="https://huggingface.co/black-forest-labs/FLUX.1-dev/media/main/dev_grid.jpg"
             alt="Flux.1"
             onerror="this.outerHTML='<div class=placeholder>🌊</div>'">
        <div class="info">
            <h3>Flux.1</h3>
            <p class="company">Black Forest Labs · 2024</p>
            <div class="tags">
                <span class="tag tag-fm">Flow Matching ✨</span>
                <span class="tag tag-imagen">Imagen</span>
            </div>
        </div>
    </div>

    <div class="model-card">
        <img src="https://cdn.openai.com/API/docs/images/sora/woman_skyline_original_720p.jpeg"
             alt="Sora 2"
             onerror="this.outerHTML='<div class=placeholder>🎬</div>'">
        <div class="info">
            <h3>Sora 2</h3>
            <p class="company">OpenAI · 2025</p>
            <div class="tags">
                <span class="tag tag-diffusion">Diffusion Transformer</span>
                <span class="tag tag-video">Video</span>
            </div>
        </div>
    </div>

    <div class="model-card">
        <img src="https://storage.googleapis.com/gweb-uniblog-publish-prod/images/AF_social_share.width-1300.jpg"
             alt="AlphaFold 3"
             onerror="this.outerHTML='<div class=placeholder>🧬</div>'">
        <div class="info">
            <h3>AlphaFold 3</h3>
            <p class="company">Google DeepMind · 2024</p>
            <div class="tags">
                <span class="tag tag-diffusion">Diffusion</span>
                <span class="tag tag-ciencia">Ciencia</span>
            </div>
        </div>
    </div>

    <div class="model-card">
        <div class="placeholder">🎨</div>
        <div class="info">
            <h3>DALL·E 3</h3>
            <p class="company">OpenAI · 2023</p>
            <div class="tags">
                <span class="tag tag-diffusion">Diffusion</span>
                <span class="tag tag-imagen">Imagen</span>
            </div>
        </div>
    </div>

    <div class="model-card">
        <div class="placeholder">🎬</div>
        <div class="info">
            <h3>Veo 3</h3>
            <p class="company">Google DeepMind · 2025</p>
            <div class="tags">
                <span class="tag tag-diffusion">Latent Diffusion</span>
                <span class="tag tag-video">Video + Audio</span>
            </div>
        </div>
    </div>

    <div class="model-card">
        <div class="placeholder">🎵</div>
        <div class="info">
            <h3>Stable Audio</h3>
            <p class="company">Stability AI · 2024</p>
            <div class="tags">
                <span class="tag tag-diffusion">Diffusion</span>
                <span class="tag tag-audio">Audio</span>
            </div>
        </div>
    </div>

    <div class="model-card">
        <div class="placeholder">🎭</div>
        <div class="info">
            <h3>Midjourney v6</h3>
            <p class="company">Midjourney · 2024</p>
            <div class="tags">
                <span class="tag tag-diffusion">Diffusion</span>
                <span class="tag tag-imagen">Imagen</span>
            </div>
        </div>
    </div>

</div>
"""

HTML(gallery_html)

---
## 0.2 ¿Cómo representamos imágenes, video y proteínas como vectores?

Todos estos modelos trabajan sobre datos muy distintos — imágenes, videos, audio, proteínas — pero comparten un insight fundamental: **todo dato puede representarse como un vector en $\mathbb{R}^d$.**

### Imágenes

Una imagen es una grilla de $H \times W$ píxeles. Cada píxel tiene 3 valores (canales RGB), cada uno entre 0 y 255 (o normalizado a $[0,1]$). Si "desenrollamos" la grilla, obtenemos un único vector:

$$x \in \mathbb{R}^{H \times W \times 3}$$

<div style="display: flex; gap: 20px; align-items: center; justify-content: center; margin: 15px 0; flex-wrap: wrap;">
  <div style="text-align: center;">
    <img src="https://upload.wikimedia.org/wikipedia/commons/2/2b/Pixel-example.png"
         alt="Ejemplo: imagen ampliada mostrando píxeles individuales"
         style="max-height: 180px; border-radius: 8px; border: 1px solid #ddd;"
         onerror="this.style.display='none'">
    <p style="font-size: 12px; color: #666; margin-top: 4px;">Cada píxel es un número (o 3 números en RGB)</p>
  </div>
  <div style="text-align: center;">
    <img src="https://upload.wikimedia.org/wikipedia/commons/3/3b/Rgb-raster-image.svg"
         alt="Imagen como grilla de píxeles RGB"
         style="max-height: 180px; border-radius: 8px; border: 1px solid #ddd;"
         onerror="this.style.display='none'">
    <p style="font-size: 12px; color: #666; margin-top: 4px;">Una imagen = grilla de valores RGB</p>
  </div>
  <div style="text-align: center;">
    <img src="https://upload.wikimedia.org/wikipedia/commons/5/56/RGB_channels_separation.png"
         alt="Descomposición de una imagen en canales R, G, B"
         style="max-height: 180px; border-radius: 8px; border: 1px solid #ddd;"
         onerror="this.style.display='none'">
    <p style="font-size: 12px; color: #666; margin-top: 4px;">Separación en canales R, G, B</p>
  </div>
</div>

**Ejemplo concreto:** una imagen de $256 \times 256$ a color es un punto en $\mathbb{R}^{196{,}608}$. Una imagen de $1024 \times 1024$ vive en $\mathbb{R}^{3{,}145{,}728}$ — más de 3 millones de dimensiones.

### Video

Un video es simplemente una secuencia de $T$ frames (imágenes):

$$x \in \mathbb{R}^{T \times H \times W \times 3}$$

<div style="display: flex; gap: 20px; align-items: center; justify-content: center; margin: 15px 0; flex-wrap: wrap;">
  <div style="text-align: center;">
    <img src="https://upload.wikimedia.org/wikipedia/commons/7/73/The_Horse_in_Motion.jpg"
         alt="Muybridge: The Horse in Motion — un video es una secuencia de frames"
         style="max-height: 200px; border-radius: 8px; border: 1px solid #ddd;"
         onerror="this.style.display='none'">
    <p style="font-size: 12px; color: #666; margin-top: 4px; max-width: 400px;">Muybridge (1878): un video es una secuencia de imágenes (frames)</p>
  </div>
  <div style="text-align: center;">
    <img src="https://upload.wikimedia.org/wikipedia/commons/d/dd/Muybridge_race_horse_animated.gif"
         alt="Muybridge horse animated — los frames forman movimiento"
         style="max-height: 200px; border-radius: 8px; border: 1px solid #ddd;"
         onerror="this.style.display='none'">
    <p style="font-size: 12px; color: #666; margin-top: 4px;">Frames → movimiento continuo</p>
  </div>
</div>

Un video de 5 segundos a 30 fps en resolución $256 \times 256$ es un vector en $\mathbb{R}^{29{,}491{,}200}$ — casi 30 millones de dimensiones.

### Proteínas

Una proteína es una cadena de $N$ aminoácidos. Cada aminoácido tiene átomos con coordenadas 3D. En la representación más simple (solo el carbono-$\alpha$ del backbone):

$$x \in \mathbb{R}^{N \times 3}$$

<div style="display: flex; gap: 20px; align-items: center; justify-content: center; margin: 15px 0; flex-wrap: wrap;">
  <div style="text-align: center;">
    <img src="https://upload.wikimedia.org/wikipedia/commons/6/60/Myoglobin.png"
         alt="Mioglobina — estructura 3D de una proteína"
         style="max-height: 200px; border-radius: 8px; border: 1px solid #ddd;"
         onerror="this.style.display='none'">
    <p style="font-size: 12px; color: #666; margin-top: 4px;">Mioglobina: cada átomo tiene coordenadas $(x, y, z)$</p>
  </div>
</div>

Esto es exactamente lo que genera **AlphaFold 3** con su módulo de difusión: coordenadas 3D de átomos.

---

> **Insight clave:** sin importar la modalidad, la tarea del modelo generativo siempre es la misma: aprender a producir nuevos puntos $x \in \mathbb{R}^d$ que "parezcan" datos reales. Cada imagen, video o proteína es una **partícula** cuya posición está dada por $d$ coordenadas.

<img src="https://github.com/llealgt/generative_AI_2023/blob/f338bec322d28cb2bbfec965577eedc2a9ecfd21/imgs/pic5_point_in_space.png?raw=true" alt="Un punto en 2D que representa un objeto" width="800">

> Como humanos estamos limitados a ver máximo 3 dimensiones, por lo cual usaremos ejemplos y analogías en 2 y 3 dimensiones donde cada punto en el espacio representa a un objeto (imagen, video, proteína, etc).

La siguiente celda visualiza esta idea para imágenes:

In [ ]:
# Visualización: una imagen es un vector en R^d
import torchvision

# Descargar una imagen de CIFAR-10
cifar = torchvision.datasets.CIFAR10(root='./data', download=True, train=True)
img_pil, label = cifar[3]  # un ciervo/deer — imagen reconocible
img_np = np.array(img_pil)  # (32, 32, 3) valores 0-255

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5),
                         gridspec_kw={'width_ratios': [1, 1.2, 1.8]})

# --- Panel 1: imagen original ---
axes[0].imshow(img_np)
axes[0].set_title(f"Imagen original\n({img_np.shape[0]}×{img_np.shape[1]}×{img_np.shape[2]})",
                  fontsize=12, weight='bold')
axes[0].axis('off')

# --- Panel 2: zoom a un patch 4x4 con valores RGB ---
patch = img_np[8:12, 14:18]  # patch 4x4 de una zona interesante
axes[1].imshow(patch, interpolation='nearest', extent=[0, 4, 0, 4])
for i in range(4):
    for j in range(4):
        r, g, b = patch[3-i, j]
        # Elegir color de texto según brillo
        brightness = 0.299*r + 0.587*g + 0.114*b
        txt_color = 'white' if brightness < 128 else 'black'
        axes[1].text(j + 0.5, i + 0.5, f'{r}\n{g}\n{b}',
                     ha='center', va='center', fontsize=7,
                     color=txt_color, weight='bold')
axes[1].set_title("Zoom: patch 4×4\n(valores RGB por píxel)", fontsize=12, weight='bold')
axes[1].set_xticks(range(5)); axes[1].set_yticks(range(5))
axes[1].grid(True, color='white', linewidth=2)
axes[1].tick_params(labelbottom=False, labelleft=False)

# --- Panel 3: vector desenrollado ---
flat = img_np.astype(np.float32).flatten() / 255.0  # (3072,)
colors_vec = np.zeros((len(flat), 3))
# Colorear R, G, B
n_pixels = 32 * 32
colors_vec[:n_pixels, 0] = flat[:n_pixels]          # canal R en rojo
colors_vec[n_pixels:2*n_pixels, 1] = flat[n_pixels:2*n_pixels]  # canal G en verde
colors_vec[2*n_pixels:, 2] = flat[2*n_pixels:]      # canal B en azul

# Mostrar como barras de color
axes[2].bar(range(len(flat)), flat, width=1.0, color=colors_vec, alpha=0.8)
axes[2].set_xlim(0, len(flat))
axes[2].set_ylim(0, 1.1)
axes[2].set_xlabel("Índice de componente", fontsize=11)
axes[2].set_ylabel("Valor (normalizado)", fontsize=11)
axes[2].set_title(r"Vector desenrollado: $x \in \mathbb{R}^{3072}$", fontsize=12, weight='bold')
# Marcadores de canales
for pos, ch in [(n_pixels//2, 'R'), (n_pixels + n_pixels//2, 'G'), (2*n_pixels + n_pixels//2, 'B')]:
    axes[2].text(pos, 1.05, ch, ha='center', fontsize=11, weight='bold',
                 color={'R': 'red', 'G': 'green', 'B': 'blue'}[ch])

fig.suptitle("Una imagen es un punto en un espacio de alta dimensión",
             fontsize=14, weight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Dimensiones de la imagen: {img_np.shape} → vector en R^{img_np.size}")
print(f"Una imagen de 256×256 sería un vector en R^{256*256*3:,}")

---
## 0.3 ¿Qué significa *generar* algo?

Ahora que sabemos que todo dato es un vector $x \in \mathbb{R}^d$, podemos formular el problema de generación de forma precisa.

<img src="https://github.com/llealgt/generative_AI_2023/blob/c2877a16f00d17f5d561925999edca3d22a6ca25/imgs/pic1_generate_something.png?raw=true" alt="¿Qué significa generar algo?" width="800">

> La pregunta *"¿qué tan buena es una imagen?"* se convierte en *"¿qué tan probable es bajo la distribución de los datos?"*

### El mundo real tiene una distribución de datos

Existe una distribución de probabilidad $p_{\text{data}}(x)$ sobre $\mathbb{R}^d$ que describe *todos* los datos posibles de un dominio. Por ejemplo:

- $p_{\text{data}}^{\text{caras}}$: la distribución de todas las fotos de caras humanas posibles
- $p_{\text{data}}^{\text{gatos}}$: la distribución de todas las fotos de gatos posibles
- $p_{\text{data}}^{\text{proteínas}}$: la distribución de todas las estructuras proteicas viables

### Un dataset es un conjunto de muestras

Nuestro dataset $\mathcal{D} = \{x_1, x_2, \ldots, x_N\}$ son **muestras finitas** extraídas de esta distribución:

$$x_i \sim p_{\text{data}}(x) \quad \text{para } i = 1, \ldots, N$$

**No tenemos una fórmula para $p_{\text{data}}$** — solo tenemos los ejemplos.

### Generar = muestrear de $p_{\text{data}}$

**Generar** un dato nuevo significa producir un $x_{\text{nuevo}} \sim p_{\text{data}}$ que:
1. Se vea como un dato real (pertenezca a la distribución)
2. **No sea una copia** de ningún $x_i$ del dataset
3. Sea una muestra *independiente* de $p_{\text{data}}$

<img src="https://github.com/llealgt/generative_AI_2023/blob/caba5ce051ded7893601b2732f40a1df5682ac27/imgs/pic2_generate_something.png?raw=true" alt="Generar = muestrear de p_data" width="800">

### ¿Por qué es difícil?

- $p_{\text{data}}$ está **concentrada** en una variedad (manifold) de dimensión muy baja dentro del enorme $\mathbb{R}^d$
- Imagina: el conjunto de "todas las fotos de gatos válidas" es una superficie increíblemente delgada dentro de $\mathbb{R}^{196{,}608}$. Un punto aleatorio en ese espacio es **ruido puro**, no un gato
- No podemos evaluar $p_{\text{data}}(x)$ directamente — es intratable
- No podemos enumerar el soporte — la dimensión es demasiado alta

La siguiente celda visualiza esta intuición en 2D:

In [ ]:
# Visualización: distribución de datos vs muestreo aleatorio
from sklearn.datasets import make_moons

np.random.seed(42)
x_data, _ = make_moons(n_samples=500, noise=0.05)
x_data = (x_data - x_data.mean(0)) / x_data.std(0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Panel izquierdo: muestras de p_data con contorno de densidad ---
from scipy.stats import gaussian_kde
kde = gaussian_kde(x_data.T)
xx, yy = np.meshgrid(np.linspace(-30, 30, 100), np.linspace(-3, 3, 100))
zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
ax1.contourf(xx, yy, zz, levels=15, cmap='Blues', alpha=0.4)
ax1.scatter(x_data[:, 0], x_data[:, 1], s=12, alpha=0.6, color='#0066cc', zorder=5)
ax1.set_title(r"500 muestras de $p_{\rm data}$" + "\n(Two Moons)", fontsize=13, weight='bold')
ax1.set_xlim(-3, 3); ax1.set_ylim(-3, 3)
ax1.set_aspect('equal')
ax1.text(0, -2.7, r"Los datos ocupan una fracción minúscula de $\mathbb{R}^2$",
         ha='center', fontsize=10, style='italic', color='#333')

# --- Panel derecho: puntos aleatorios uniformes ---
x_random = np.random.uniform(-15, 15, size=(500, 2))
ax2.scatter(x_random[:, 0], x_random[:, 1], s=12, alpha=0.4, color='#999')
ax2.set_title(r"500 puntos aleatorios en $\mathbb{R}^2$" + "\n(muestreo uniforme)",
              fontsize=13, weight='bold')
ax2.set_xlim(-15, 15); ax2.set_ylim(-15, 15)
ax2.set_aspect('equal')
# Rectángulo mostrando la región donde vive p_data
from matplotlib.patches import Rectangle
rect = Rectangle((-3, -3), 6, 6, linewidth=1.5, edgecolor='#0066cc',
                 facecolor='#0066cc', alpha=0.08, linestyle='--', zorder=4)
ax2.add_patch(rect)
ax2.annotate(r"Región de $p_{\rm data}$", xy=(3, 3), xytext=(6, 10),
             fontsize=9, color='#0066cc', weight='bold',
             arrowprops=dict(arrowstyle='->', color='#0066cc', lw=1.2))

fig.suptitle("¿Por qué es difícil generar? Los datos viven en una región muy pequeña del espacio",
             fontsize=13, weight='bold', y=1.02)
plt.tight_layout()
plt.show()

La pregunta "que tan buena es una imágen?" se convierte en "que tan probable es bajo la distribución de los datos"

---
## 0.4 Generación condicional: "dame un gato"

En la práctica, no solo queremos generar *cualquier* muestra de $p_{\text{data}}$ — queremos **controlar** qué se genera.

### De incondicional a condicional

En vez de muestrear de $p(x)$ (cualquier imagen), queremos muestrear de una distribución **condicional**:

$$x \sim p(x \mid c)$$

donde $c$ es una **condición** que especifica qué tipo de dato queremos.

### ¿Qué puede ser $c$?

| Tarea | Condición $c$ | Dato generado $x$ |
|-------|---------------|-------------------|
| Texto → imagen | *"A cat riding a bicycle"* | Imagen $256 \times 256$ |
| Clase → dígito | $c = 7$ (etiqueta MNIST) | Imagen de un "7" |
| Super-resolución | Imagen baja resolución | Imagen alta resolución |
| Inpainting | Imagen con región enmascarada | Imagen completa |
| Estructura de proteína | Secuencia de aminoácidos | Coordenadas 3D |

<img src="https://github.com/llealgt/generative_AI_2023/blob/64ccbb3c60216bc50198388facbe37d5f4f956a4/imgs/pic3_cond_generation.png?raw=true" alt="Generación condicional" width="800">

### Lo que NO cambia

El problema fundamental **sigue siendo el mismo**: necesitamos muestrear de una distribución en $\mathbb{R}^d$. La condición $c$ simplemente nos dice *de cuál sub-distribución* muestrear.

> En la **sección 10** veremos cómo se implementa esto en la práctica con **Classifier-Free Guidance (CFG)**. Por ahora, basta con saber que el objetivo es aprender $p(x \mid c)$.

La siguiente celda ilustra la diferencia entre generación incondicional y condicional usando una distribución 2D:

In [ ]:
# Visualización: generación incondicional vs condicional (2D)
np.random.seed(0)
k = 8  # número de clases
angles = np.linspace(0, 2 * np.pi, k, endpoint=False)
centers = 3.0 * np.stack([np.cos(angles), np.sin(angles)], axis=1)
n_per_class = 80
class_colors = plt.cm.tab10(np.arange(k))
class_names = [f"clase {i}" for i in range(k)]

# Generar puntos por clase
all_pts = []
all_labels = []
for i in range(k):
    pts = centers[i] + 0.3 * np.random.randn(n_per_class, 2)
    all_pts.append(pts)
    all_labels.extend([i] * n_per_class)
all_pts = np.concatenate(all_pts)
all_labels = np.array(all_labels)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Panel izquierdo: incondicional p(x) ---
for i in range(k):
    mask = all_labels == i
    ax1.scatter(all_pts[mask, 0], all_pts[mask, 1], s=15,
                color=class_colors[i], alpha=0.7, label=class_names[i])
ax1.set_title(r"Incondicional: $p(x)$" + "\n(todas las clases)", fontsize=13, weight='bold')
ax1.set_xlim(-5.5, 5.5); ax1.set_ylim(-5.5, 5.5)
ax1.set_aspect('equal')
ax1.legend(fontsize=8, ncol=2, loc='lower right')

# --- Panel derecho: condicional p(x | c=3) ---
target_class = 3
for i in range(k):
    mask = all_labels == i
    if i == target_class:
        ax2.scatter(all_pts[mask, 0], all_pts[mask, 1], s=25,
                    color=class_colors[i], alpha=0.9, zorder=5,
                    label=f"clase {i} (seleccionada)")
        # Círculo de énfasis
        circle = plt.Circle(centers[i], 1.0, fill=False,
                            edgecolor=class_colors[i], linewidth=2, linestyle='--')
        ax2.add_patch(circle)
    else:
        ax2.scatter(all_pts[mask, 0], all_pts[mask, 1], s=10,
                    color=(0.82, 0.82, 0.82), alpha=0.3)

ax2.set_title(r"Condicional: $p(x \mid c = 3)$" + "\n(solo una clase)",
              fontsize=13, weight='bold')
ax2.set_xlim(-5.5, 5.5); ax2.set_ylim(-5.5, 5.5)
ax2.set_aspect('equal')
ax2.legend(fontsize=10, loc='lower right')

fig.suptitle("Generación condicional = muestrear de una sub-distribución específica",
             fontsize=13, weight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 0.5 Modelos Generativos e Inteligencia Artificial Generativa

<img src="https://github.com/llealgt/generative_AI_2023/blob/e71d316db5522a3d57eb7f6e117345a0ae3cff01/imgs/pic6_gen_modeling.png?raw=true" alt="La idea de un vistazo" width="800">

En la ilustración el modelo generativo se ve como una caja, hay muchos tipos de modelos y algoritmos, como flow matching y diffusion son solo 2 tipos, como funcionan?

---
## 0.6 La idea de un vistazo: la estrategia



Ahora que sabemos **qué queremos** (muestrear de $p_{\text{data}}$ o de $p(x|c)$) y **por qué es difícil** (espacio de altísima dimensión, distribución desconocida), veamos la estrategia que usan **Diffusion** y **Flow Matching** para resolver el problema:

> Empezar con una distribución simple que sí sabemos muestrear ($\mathcal{N}(0,I)$) y aprender una **transformación continua** que la lleve a $p_{\text{data}}$.

Esta es la imagen mental que vamos a construir:

In [ ]:
# Diagrama de visión general: noise -> vector field -> data
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.set_xlim(0, 13); ax.set_ylim(0, 5); ax.axis("off")

# Distribución de ruido (izquierda)
np.random.seed(1)
noise = np.random.randn(300, 2) * 0.4
ax.scatter(1.5 + noise[:,0], 2.5 + noise[:,1], s=8, alpha=0.5, color="#888")
ax.text(1.5, 0.5, r"$x_0 \sim \mathcal{N}(0,I)$", ha="center", fontsize=14)
ax.text(1.5, 4.5, "Ruido (simple)", ha="center", fontsize=12, weight="bold")

# Caja del modelo (centro)
box = FancyBboxPatch((4.5, 1.6), 4, 1.8, boxstyle="round,pad=0.1",
                     facecolor="#ffe6cc", edgecolor="#cc6600", linewidth=2)
ax.add_patch(box)
ax.text(6.5, 2.8, r"Campo vectorial $v_\theta(x,t)$", ha="center", fontsize=13, weight="bold")
ax.text(6.5, 2.2, r"(o score $s_\theta(x,t)$)", ha="center", fontsize=11)
ax.text(6.5, 4.5, "Red neuronal", ha="center", fontsize=12, weight="bold")
ax.text(6.5, 0.5, r"$\frac{dx}{dt} = v_\theta(x,t)$  ó  $dx = f\,dt + g\,dW$",
        ha="center", fontsize=12)

# Distribución de datos (derecha) - dos lunas
t_moons = np.linspace(0, np.pi, 150)
moon1 = np.stack([np.cos(t_moons), np.sin(t_moons)], axis=1) * 0.7
moon2 = np.stack([1 - np.cos(t_moons), 1 - np.sin(t_moons) - 0.5], axis=1) * 0.7
moons = np.concatenate([moon1, moon2]) + np.array([11.0, 2.2])
moons += 0.03 * np.random.randn(*moons.shape)
ax.scatter(moons[:,0], moons[:,1], s=8, alpha=0.7, color="#0066cc")
ax.text(11.5, 0.5, r"$x_1 \sim p_{\rm data}$", ha="center", fontsize=14)
ax.text(11.5, 4.5, "Datos (complejos)", ha="center", fontsize=12, weight="bold")

# Flechas
ax.annotate("", xy=(4.4, 2.5), xytext=(2.7, 2.5),
            arrowprops=dict(arrowstyle="->", lw=2, color="#444"))
ax.annotate("", xy=(10.3, 2.5), xytext=(8.6, 2.5),
            arrowprops=dict(arrowstyle="->", lw=2, color="#444"))
ax.text(3.55, 2.85, "input", ha="center", fontsize=10, style="italic")
ax.text(9.45, 2.85, "integrar", ha="center", fontsize=10, style="italic")
ax.text(9.45, 2.15, "(Euler / ME)", ha="center", fontsize=9, style="italic")

plt.title("Modelado generativo como transporte: una red aprende cómo mover ruido a datos",
          fontsize=13, pad=20)
plt.show()


**En este taller vas a implementar, desde cero, los mismos algoritmos fundamentales que hacen funcionar todos estos modelos:**

| Lo que construirás | Técnica | Modelo real que la usa |
|---|---|---|
| Flow Matching en 2D y sobre imágenes | Caminos rectos + campo de velocidades | **Flux**, Stable Diffusion 3 |
| DDPM (Denoising Diffusion) | Score matching + proceso de denoising | **DALL·E 3**, Midjourney |
| Latent Diffusion | Diffusion en espacio latente (con autoencoder) | **Stable Diffusion**, **Sora** |
| Classifier-Free Guidance | Generación condicionada sin clasificador externo | Prácticamente **todos** los anteriores |

La diferencia entre estos modelos de frontera y lo que haremos aquí es **escala** (más datos, más parámetros, más GPUs), no los principios fundamentales. Comencemos. 👇

---
# 1. Modelado generativo como muestreo de una distribución

## 1.1 Empecemos en 2D

Ya establecimos el problema: tenemos muestras de $p_{\text{data}}$ y queremos generar nuevas. La estrategia será **transportar** una distribución simple ($\mathcal{N}(0,I)$) a $p_{\text{data}}$ usando un campo vectorial aprendido.

Antes de atacar imágenes, trabajamos en $\mathbb{R}^2$ donde **podemos visualizar todo**: la distribución, las trayectorias y el campo vectorial. Usaremos tres datasets clásicos:

In [ ]:
from sklearn.datasets import make_moons

def sample_moons(n):
    x, _ = make_moons(n_samples=n, noise=0.05)
    x = (x - x.mean(0)) / x.std(0)
    return torch.tensor(x, dtype=torch.float32)

def sample_gaussian_mix(n, k=8, r=3.0, std=0.15):
    angles = torch.linspace(0, 2*math.pi, k+1)[:-1]
    centers = torch.stack([r*torch.cos(angles), r*torch.sin(angles)], dim=1)
    idx = torch.randint(0, k, (n,))
    return centers[idx] + std * torch.randn(n, 2)

def sample_spiral(n):
    t = torch.rand(n) * 4*math.pi + math.pi/2
    x = torch.stack([t*torch.cos(t), t*torch.sin(t)], dim=1) / 12
    return x + 0.05*torch.randn_like(x)

datasets = {"Two Moons": sample_moons(3000),
            "8 Gaussians": sample_gaussian_mix(3000),
            "Spiral": sample_spiral(3000)}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, data) in zip(axes, datasets.items()):
    ax.scatter(data[:,0], data[:,1], s=4, alpha=0.6)
    ax.set_title(f"$p_{{\\rm data}}$: {name}")
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## 1.2 Anticipo animado: ¿qué queremos lograr?

Aún sin entrenar nada, podemos visualizar el objetivo: arrancamos con partículas en $\mathcal{N}(0,I)$ y queremos llevarlas a $p_{\text{data}}$. La animación de abajo hace **interpolación lineal** entre cada partícula y un punto de los datos. Esto es exactamente la trayectoria que Flow Matching aprende a seguir.


In [ ]:
# Animación teaser: linear interpolation noise -> moons
N = 500
torch.manual_seed(42)
x0 = torch.randn(N, 2)
x1 = sample_moons(N)

T_frames = 40
ts = np.linspace(0, 1, T_frames)
positions = [(1-t)*x0.numpy() + t*x1.numpy() for t in ts]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
title = ax.set_title("t = 0.00")
scat = ax.scatter(positions[0][:,0], positions[0][:,1], s=10, alpha=0.7,
                  c=np.arctan2(x1[:,1], x1[:,0]), cmap="hsv")

def update(i):
    scat.set_offsets(positions[i])
    title.set_text(f"t = {ts[i]:.2f}")
    return scat, title

anim = animation.FuncAnimation(fig, update, frames=T_frames, interval=80, blit=False)
plt.close()
HTML(anim.to_jshtml())


☝️ Esa interpolación lineal **es** el "probability path" que Flow Matching va a aprender a seguir. La pregunta es: ¿podemos hacer que una red neuronal *no necesite saber el destino $x_1$ de cada partícula* y aún así produzca este flujo? (Spoiler: sí.)


## 1.3 Las dos direcciones: destruir y crear

Antes de ver un modelo entrenado, necesitamos una intuición fundamental.

**Imagina una gota de tinta en agua quieta.** Con el tiempo, la tinta se difunde: la estructura se pierde y el color se distribuye uniformemente. Esto es **fácil** — solo hay que esperar.

Ahora imagina el proceso **al revés**: empezar con agua uniformemente teñida y *concentrar* la tinta de vuelta en la gota original. Eso parece imposible... pero es exactamente lo que hace un modelo de difusión.

| Dirección | ¿Qué hace? | ¿Fácil o difícil? |
|-----------|-----------|-------------------|
| **Forward** (datos → ruido) | Añadir ruido progresivamente | Trivial — no necesita aprender nada |
| **Reverse** (ruido → datos) | Quitar ruido paso a paso | Difícil — requiere una red neuronal |

La animación anterior mostró el camino ruido → datos como una interpolación directa. Ahora veamos el camino *contrario*: **datos → ruido**.

In [ ]:
# Animación del proceso forward conceptual: datos → ruido
# Usamos mezcla lineal simple (sin schedule DDPM — eso viene en la sección 6)
N_fwd = 800
torch.manual_seed(42)
x_data_fwd = sample_moons(N_fwd)
noise_fwd = torch.randn_like(x_data_fwd)

T_frames = 40
ts_fwd = np.linspace(0, 1, T_frames)
# Mezcla lineal: a t=0 tenemos datos puros, a t=1 tenemos ruido puro
positions_fwd = [((1 - t) * x_data_fwd + t * noise_fwd).numpy() for t in ts_fwd]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
colors_fwd = np.arctan2(x_data_fwd[:, 1].numpy(), x_data_fwd[:, 0].numpy())
scat = ax.scatter(positions_fwd[0][:, 0], positions_fwd[0][:, 1],
                  s=8, alpha=0.6, c=colors_fwd, cmap="hsv")
title = ax.set_title("Forward: t=0.00  (datos puros)", fontsize=13)

def update(i):
    scat.set_offsets(positions_fwd[i])
    label = "datos puros" if ts_fwd[i] < 0.05 else ("ruido puro" if ts_fwd[i] > 0.95 else "")
    title.set_text(f"Forward: t={ts_fwd[i]:.2f}  {label}")
    return scat, title

anim_fwd_intro = animation.FuncAnimation(fig, update, frames=T_frames,
                                          interval=80, blit=False)
plt.close()
HTML(anim_fwd_intro.to_jshtml())

**Observa:** los colores te permiten seguir cada partícula individual. La estructura de las dos lunas se *disuelve* gradualmente hasta ser indistinguible de ruido Gaussiano $\mathcal{N}(0,I)$.

### ¿Y si pudiéramos revertirlo?

El proceso forward es trivial (solo mezclamos ruido). Pero contiene una pista crucial: **si conociéramos la dirección inversa en cada paso**, podríamos arrancar desde ruido puro y reconstruir los datos.

```
  datos (t=0)  ──── forward (fácil) ────▶  ruido (t=1)
       ◄──── reverse (¿?) ────
```

La pregunta del millón: **¿quién nos dice hacia dónde ir en cada paso del reverse?**

> Respuesta: una **red neuronal** entrenada para predecir cuánto ruido hay en cada punto $x_t$ y en cada instante $t$. Eso es, en esencia, un **modelo de difusión**.

In [ ]:
# Snapshots del proceso forward en 2D
snap_times = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
fig, axes = plt.subplots(1, len(snap_times), figsize=(15, 2.8))

for ax, t in zip(axes, snap_times):
    xt = ((1 - t) * x_data_fwd + t * noise_fwd).numpy()
    ax.scatter(xt[:, 0], xt[:, 1], s=3, alpha=0.5, c=colors_fwd, cmap="hsv")
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"t = {t:.1f}", fontsize=11)

axes[0].set_ylabel("Forward →", fontsize=11, color="steelblue")
fig.suptitle("Proceso forward: datos → ruido", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Resumen hasta aquí:**

1. **Muestreo generativo** = transportar partículas desde $\mathcal{N}(0,I)$ hacia $p_{\text{data}}$
2. El **proceso forward** (datos → ruido) es trivial y no requiere aprendizaje
3. El **proceso reverse** (ruido → datos) es el que necesita un modelo — y es exactamente lo que hacen tanto *Flow Matching* como *Diffusion Models*

La diferencia entre estos dos enfoques (y las fórmulas precisas del forward) las veremos más adelante. Por ahora, lo importante es la intuición: **si existe un camino de datos a ruido, existe uno de ruido a datos.**

Veamos qué pasa cuando una red neuronal *ya entrenada* hace exactamente esto sobre imágenes reales. 👇

---
# 2. Sampling con un modelo pre-entrenado

Antes de la teoría, veamos **qué hace** un modelo de difusión real. Cargaremos un DDPM pre-entrenado en CIFAR-10 ($32\times 32$) desde Hugging Face y observaremos cómo transforma ruido puro en una imagen.

$$
x_T \;\underset{\text{denoising}}{\xrightarrow{\hspace{2cm}}}\; x_{T-1} \;\xrightarrow{\hspace{1cm}}\; \cdots \;\xrightarrow{\hspace{1cm}}\; x_0
$$

donde $x_T \sim \mathcal{N}(0,I)$ y $x_0 \sim p_{\text{data}}$.


In [ ]:
from diffusers import DDPMScheduler, UNet2DModel

# DDPM pre-entrenado en CIFAR-10
model_id = "google/ddpm-cifar10-32"
unet = UNet2DModel.from_pretrained(model_id).to(device)
scheduler = DDPMScheduler.from_pretrained(model_id)
print(f"U-Net params: {sum(p.numel() for p in unet.parameters())/1e6:.1f}M")


In [ ]:
@torch.no_grad()
def sample_ddpm(unet, scheduler, num_inference_steps=50, batch_size=4):
    scheduler.set_timesteps(num_inference_steps)
    x = torch.randn(batch_size, 3, 32, 32, device=device)
    trajectory = [x.cpu().clone()]
    for i, t in enumerate(scheduler.timesteps):
        noise_pred = unet(x, t).sample
        x = scheduler.step(noise_pred, t, x).prev_sample
        trajectory.append(x.cpu().clone())
    return x, trajectory

unet.eval()
samples_pre, traj_pre = sample_ddpm(unet, scheduler, num_inference_steps=100, batch_size=4)
print(f"Guardados {len(traj_pre)} snapshots de la trayectoria")

In [ ]:
# Visualización estática: la grilla de la trayectoria
def to_img(x):
    return ((x.clamp(-1,1) + 1) / 2).permute(0,2,3,1).numpy()

snaps = [0, 20, 40, 60, 80, 100]
fig, axes = plt.subplots(4, len(snaps), figsize=(len(snaps)*1.4, 5))
for j in range(4):
    for i, k in enumerate(snaps):
        axes[j, i].imshow(to_img(traj_pre[k])[j]); axes[j, i].axis("off")
        if j == 0: axes[j, i].set_title(f"step {k}", fontsize=10)
plt.suptitle("Trayectoria de denoising (de ruido $x_T$ a imagen $x_0$)", y=1.02)
plt.tight_layout(); plt.show()

## 2.1 La misma trayectoria, animada

Ahora animamos la transición completa: cómo el ruido puro se "esculpe" en una imagen reconocible.


In [ ]:
# Animación: trayectoria de denoising CIFAR-10
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
ims = []
for j, ax in enumerate(axes):
    ax.axis("off")
    im = ax.imshow(to_img(traj_pre[0])[j])
    ims.append(im)
title = fig.suptitle("step 0 — ruido puro", fontsize=12)

def update(i):
    for j, im in enumerate(ims):
        im.set_data(to_img(traj_pre[i])[j])
    title.set_text(f"step {i}/{len(traj_pre)-1}")
    return ims + [title]

anim = animation.FuncAnimation(fig, update, frames=len(traj_pre), interval=120, blit=False)
plt.close()
HTML(anim.to_jshtml())


**Observa:** las primeras frames son ruido puro; en las del medio aparecen *manchas de color con estructura espacial*; en las últimas, formas reconocibles. La partícula recorrió un **camino continuo** en el espacio de píxeles guiada por el campo vectorial aprendido.

**Que pasa si tu caso de uso necesita trabajar con datos propios y/o privados?** Por ejemplo: generar imágenes publicitarias USANDO PRODUCTOS PROPIOS DE UNA EMPRESA. Un ML/AI engineer debe ser capaz no solo de usar modelos de otros si no de entrenar modelos propios.

Ahora vamos a la teoría para entender **qué calcula** el modelo y **cómo** lo entrenaríamos nosotros.


---
# 3. Refresher: ODEs y campos vectoriales

Antes de saltar a probability paths abstractos, hagamos un detour por los conceptos básicos con ejemplos **analíticos** y **visualizables**.

## 3.1 ¿Qué es una ODE?

Una **ecuación diferencial ordinaria** (ODE) describe cómo evoluciona una variable en el tiempo dado su estado actual:

$$
\frac{dx}{dt} = v(x, t)
$$

- $x(t) \in \mathbb{R}^d$ es la **posición** de una partícula.
- $v(x,t)$ es el **campo vectorial**: una flecha en cada punto que dice hacia dónde moverse.

> **Analogía física directa:** es exactamente como una corriente en un río. El campo $v(x,t)$ son las velocidades del agua en cada punto. Una hoja flotando sigue las trayectorias dictadas por la corriente.

## 3.2 El método de Euler

Si conocemos $v(x,t)$ y la condición inicial $x(0) = x_0$, podemos integrar la ODE numéricamente. El método más simple es **Euler**: 

$$
x(t + \Delta t) \approx x(t) + v(x(t), t)\,\Delta t.
$$

Con $\Delta t$ pequeño, esto aproxima la trayectoria real.


## 3.3 Ejemplo 1: campo rotacional

Un primer ejemplo concreto: $v(x,y) = (-y,\, x)$. Cada partícula es empujada **perpendicular** a su vector posición. ¿Adivinas qué trayectoria sigue? (Spoiler: círculos centrados en el origen.)


In [ ]:
# Ejemplo 1: campo rotacional v(x,y) = (-y, x)
def v_rotation(x, t=None):
    return torch.stack([-x[:,1], x[:,0]], dim=1)

# Visualizamos el campo
grid = 18
xs = torch.linspace(-2.5, 2.5, grid); ys = torch.linspace(-2.5, 2.5, grid)
X, Y = torch.meshgrid(xs, ys, indexing="xy")
pts = torch.stack([X.flatten(), Y.flatten()], dim=1)
V = v_rotation(pts).numpy()

# Integramos algunas trayectorias con Euler
def integrate_euler(v_fn, x0, T=2*math.pi, n_steps=120):
    x = x0.clone()
    traj = [x.clone()]
    dt = T / n_steps
    for _ in range(n_steps):
        x = x + v_fn(x) * dt
        traj.append(x.clone())
    return torch.stack(traj, dim=0)  # (n_steps+1, N, 2)

n_particles = 8
angles = torch.linspace(0, 2*math.pi, n_particles+1)[:-1]
x0 = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1) * 1.5
traj_rot = integrate_euler(v_rotation, x0, T=2*math.pi, n_steps=120)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
# Subplot 1: el campo vectorial
axes[0].quiver(X.numpy(), Y.numpy(), V[:,0].reshape(grid,grid), V[:,1].reshape(grid,grid),
               color="steelblue", scale=30, alpha=0.7)
axes[0].set_title(r"Campo vectorial $v(x,y) = (-y,\,x)$")
axes[0].set_aspect("equal"); axes[0].set_xticks([]); axes[0].set_yticks([])

# Subplot 2: trayectorias
axes[1].quiver(X.numpy(), Y.numpy(), V[:,0].reshape(grid,grid), V[:,1].reshape(grid,grid),
               color="lightgray", scale=30, alpha=0.5)
for i in range(n_particles):
    axes[1].plot(traj_rot[:, i, 0], traj_rot[:, i, 1], lw=1.5)
    axes[1].scatter(traj_rot[0, i, 0], traj_rot[0, i, 1], s=40, color="red", zorder=5)
axes[1].set_title("Trayectorias (Euler, 120 pasos)\nLas partículas giran en círculos")
axes[1].set_aspect("equal"); axes[1].set_xticks([]); axes[1].set_yticks([])
plt.tight_layout(); plt.show()


## 3.4 Animación: partículas en el campo rotacional


In [ ]:
# Animación del campo rotacional
n_anim = 30
torch.manual_seed(7)
x0_anim = 2 * torch.rand(n_anim, 2) - 1  # ruido [-1,1]
traj_anim = integrate_euler(v_rotation, x0_anim, T=2*math.pi, n_steps=80)
T_frames = traj_anim.shape[0]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.quiver(X.numpy(), Y.numpy(), V[:,0].reshape(grid,grid), V[:,1].reshape(grid,grid),
          color="lightgray", scale=30, alpha=0.4)
scat = ax.scatter(traj_anim[0,:,0], traj_anim[0,:,1], s=40,
                  c=np.arange(n_anim), cmap="hsv", zorder=5, edgecolor="black")
ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5); ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
title = ax.set_title("Partículas siguiendo el campo rotacional")

def update(i):
    scat.set_offsets(traj_anim[i].numpy())
    return scat, title

anim_rot = animation.FuncAnimation(fig, update, frames=T_frames, interval=60, blit=False)
plt.close()
HTML(anim_rot.to_jshtml())


## 3.5 Ejemplo 2: campo "sink" (atractor al origen)

Ahora un campo que mueve todo hacia el origen: $v(x,y) = -x$. Las partículas convergen a $(0,0)$. Esta es la idea de un *atractor*.


In [ ]:
# Ejemplo 2: campo sink v(x,y) = -x  (todo decae al origen)
def v_sink(x, t=None):
    return -x

V_sink = v_sink(pts).numpy()

torch.manual_seed(3)
x0_sink = 3 * torch.rand(n_anim, 2) - 1.5
traj_sink = integrate_euler(v_sink, x0_sink, T=3.0, n_steps=80)
T_frames = traj_sink.shape[0]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].quiver(X.numpy(), Y.numpy(), V_sink[:,0].reshape(grid,grid), V_sink[:,1].reshape(grid,grid),
               color="darkred", scale=30, alpha=0.7)
axes[0].set_title(r"Campo $v(x,y) = -x$ (sink)")
axes[0].set_aspect("equal"); axes[0].set_xticks([]); axes[0].set_yticks([])

axes[1].quiver(X.numpy(), Y.numpy(), V_sink[:,0].reshape(grid,grid), V_sink[:,1].reshape(grid,grid),
               color="lightgray", scale=30, alpha=0.5)
for i in range(n_anim):
    axes[1].plot(traj_sink[:, i, 0], traj_sink[:, i, 1], lw=1, alpha=0.5)
axes[1].scatter(traj_sink[0,:,0], traj_sink[0,:,1], s=20, color="red", label="$t=0$")
axes[1].scatter(traj_sink[-1,:,0], traj_sink[-1,:,1], s=20, color="green", label="$t=T$")
axes[1].legend()
axes[1].set_title("Convergencia hacia el origen")
axes[1].set_aspect("equal"); axes[1].set_xticks([]); axes[1].set_yticks([])
plt.tight_layout(); plt.show()


## 3.6 Ejemplo 3: un campo dependiente del tiempo

En Flow Matching y Diffusion, el campo vectorial **depende de $t$** ($v_t(x)$): cambia a lo largo del tiempo. Veamos un caso simple — un campo que primero contrae y luego rota:


In [ ]:
# Campo dependiente del tiempo: v(x,t) = -alpha(t) * x + beta(t) * R x
def v_time(x, t):
    # t: scalar tensor or float
    if isinstance(t, torch.Tensor): t_val = t.item() if t.ndim == 0 else float(t[0])
    else: t_val = float(t)
    alpha = 1.0 - t_val  # contracción decrece con t
    beta = t_val          # rotación crece con t
    Rx = torch.stack([-x[:,1], x[:,0]], dim=1)
    return -alpha * x + beta * Rx

# Integramos con un campo que cambia en el tiempo
def integrate_time(v_fn, x0, T=2.0, n_steps=120):
    x = x0.clone()
    traj = [x.clone()]
    dt = T / n_steps
    for k in range(n_steps):
        t = (k * dt) / T   # tiempo normalizado
        x = x + v_fn(x, t) * dt
        traj.append(x.clone())
    return torch.stack(traj, dim=0)

torch.manual_seed(11)
x0_t = 2.5 * (torch.rand(40, 2) - 0.5)
traj_t = integrate_time(v_time, x0_t, T=4.0, n_steps=120)

# Mostramos el campo en 3 instantes
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, t_val in zip(axes, [0.0, 0.5, 1.0]):
    V_t = v_time(pts, t_val).numpy()
    ax.quiver(X.numpy(), Y.numpy(), V_t[:,0].reshape(grid,grid), V_t[:,1].reshape(grid,grid),
              color="purple", scale=30, alpha=0.7)
    ax.set_title(f"$v(x, t={t_val:.1f})$")
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Un campo vectorial que cambia con el tiempo", y=1.02)
plt.tight_layout(); plt.show()


**Esto es exactamente lo que aprende Flow Matching**: una función $v_\theta(x, t)$ que produce un campo vectorial **distinto en cada $t$**, y al integrarla con Euler obtenemos las trayectorias que llevan ruido a datos.

Con esta intuición fresca, ya estamos listos para la teoría.


---
# 4. Probability paths, vector fields y el truco de Flow Matching

## 4.1 La idea geométrica

Definimos un **camino de probabilidades** (probability path) $p_t(x)$ con $t\in [0,1]$ que **interpola** entre dos distribuciones:

$$
p_0(x) = \mathcal{N}(x;0,I) \quad\text{(prior simple)}, \qquad p_1(x) = p_{\text{data}}(x).
$$

Si pudiéramos transportar partículas $x \sim p_0$ de modo que en $t=1$ estén distribuidas como $p_1$, ¡resuelto!

## 4.2 La conexión con física: ecuación de continuidad

Si las partículas se mueven con velocidad $v_t(x)$, la densidad satisface la **ecuación de continuidad** (conservación de masa, idéntica a la mecánica de fluidos):

$$
\frac{\partial p_t}{\partial t} + \nabla \cdot (p_t \, v_t) = 0.
$$

Dado un camino $p_t$, existe (al menos un) campo $v_t$ que lo genera. **Aprender ese campo vectorial** es Flow Matching.

## 4.3 ¿Qué camino elegir?

El truco más simple: **interpolar linealmente** entre una muestra de ruido $x_0\sim\mathcal{N}(0,I)$ y una muestra de datos $x_1\sim p_{\text{data}}$:

$$
x_t = (1-t)\, x_0 + t\, x_1, \qquad t \in [0,1].
$$

La velocidad de esa partícula condicional es trivialmente:
$$
\frac{d x_t}{dt} = x_1 - x_0.
$$

## 4.4 Conditional Flow Matching (CFM): el "truco"

No conocemos el campo *marginal* $v_t(x)$, pero **sí conocemos el condicional**: dado un par $(x_0, x_1)$, la velocidad es $x_1 - x_0$.

El paper de **Lipman et al. 2023** demuestra que entrenar para predecir esta velocidad condicional **da el mismo gradiente** que entrenar para el marginal:

$$
\boxed{\;\mathcal{L}_{\text{CFM}} = \mathbb{E}_{t,\, x_0,\, x_1}\;\big\| v_\theta(x_t, t) - (x_1 - x_0) \big\|^2\;}
$$

con $t\sim\mathcal{U}(0,1)$, $x_0\sim\mathcal{N}(0,I)$, $x_1 \sim p_{\text{data}}$, $x_t=(1-t)x_0+t x_1$.

**Eso es todo el entrenamiento.** Una loss de MSE.


---
# 5. Flow Matching en 2D — *manos a la obra*

Implementemos el modelo más simple posible: un MLP que predice $v_\theta(x,t)$ para puntos en $\mathbb{R}^2$.


In [ ]:
class TimeMLP(nn.Module):
    '''MLP que recibe (x, t) y devuelve un vector de la misma dimensión que x.'''
    def __init__(self, dim=2, hidden=128, t_embed=32):
        super().__init__()
        self.t_embed_dim = t_embed
        self.net = nn.Sequential(
            nn.Linear(dim + t_embed, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, dim),
        )

    def sinusoidal(self, t):
        # Embedding sinusoidal estándar (igual idea que en Transformers)
        half = self.t_embed_dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t[:, None] * freqs[None] * 2 * math.pi
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

    def forward(self, x, t):
        if t.ndim == 0:
            t = t.expand(x.size(0))
        emb = self.sinusoidal(t)
        return self.net(torch.cat([x, emb], dim=-1))


In [ ]:
# Elegimos el dataset (puedes cambiar a "8 Gaussians" o "Spiral")
DATA_NAME = "Two Moons"
data_sampler = {
    "Two Moons": sample_moons,
    "8 Gaussians": sample_gaussian_mix,
    "Spiral": sample_spiral,
}[DATA_NAME]

# --- Entrenamiento de Flow Matching ---
model_fm = TimeMLP(dim=2).to(device)
opt = torch.optim.Adam(model_fm.parameters(), lr=2e-3)

n_steps = 5000; batch_size = 512
losses = []
for step in range(n_steps):
    x1 = data_sampler(batch_size).to(device)
    x0 = torch.randn_like(x1)
    t  = torch.rand(batch_size, device=device)
    xt = (1 - t[:, None]) * x0 + t[:, None] * x1
    v_target = x1 - x0
    v_pred = model_fm(xt, t)
    loss = F.mse_loss(v_pred, v_target)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if (step+1) % 1000 == 0:
        print(f"step {step+1:>5d}  loss = {loss.item():.4f}")

plt.figure(figsize=(5,2.5))
plt.plot(losses); plt.title("Loss de Flow Matching"); plt.xlabel("step"); plt.ylabel("MSE"); plt.show()


## 5.1 Muestreo: integrando la ODE con Euler

$$
\frac{dx}{dt} = v_\theta(x, t), \quad x(0) \sim \mathcal{N}(0,I) \quad\Longrightarrow\quad x_{t+\Delta t} = x_t + v_\theta(x_t, t)\,\Delta t.
$$


In [ ]:
@torch.no_grad()
def euler_sample(model, n_samples=2000, n_steps=100, dim=2, return_traj=False):
    x = torch.randn(n_samples, dim, device=device)
    dt = 1.0 / n_steps
    traj = [x.cpu().clone()]
    for k in range(n_steps):
        t = torch.full((n_samples,), k*dt, device=device)
        v = model(x, t)
        x = x + v * dt
        traj.append(x.cpu().clone())
    return (x.cpu(), traj) if return_traj else x.cpu()

samples, traj_fm = euler_sample(model_fm, n_samples=2000, n_steps=100, return_traj=True)
real = data_sampler(2000)

fig, axes = plt.subplots(1, 2, figsize=(9,4.2))
axes[0].scatter(real[:,0], real[:,1], s=4, alpha=0.5); axes[0].set_title("Datos reales")
axes[1].scatter(samples[:,0], samples[:,1], s=4, alpha=0.5, color="C1")
axes[1].set_title("Muestras Flow Matching (Euler, 100 pasos)")
for a in axes: a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
plt.show()


## 5.2 Animación: el campo vectorial aprendido en distintos tiempos

Tres snapshots del campo $v_\theta(x,t)$ aprendido. En $t\approx 0$ apunta hacia las modas; en $t\approx 1$ es casi nulo (los puntos ya están en su sitio).


In [ ]:
@torch.no_grad()
def plot_vector_field(model, t_value, ax, lim=3.0, grid=18):
    xs = torch.linspace(-lim, lim, grid); ys = torch.linspace(-lim, lim, grid)
    Xg, Yg = torch.meshgrid(xs, ys, indexing="xy")
    pts = torch.stack([Xg.flatten(), Yg.flatten()], dim=1).to(device)
    t = torch.full((pts.size(0),), t_value, device=device)
    v = model(pts, t).cpu().numpy()
    ax.quiver(Xg.numpy(), Yg.numpy(), v[:,0].reshape(grid,grid), v[:,1].reshape(grid,grid),
              scale=40, alpha=0.7, color="C0")
    ax.set_title(f"$v_\\theta(x, t={t_value:.2f})$"); ax.set_aspect("equal")
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, t_val in zip(axes, [0.0, 0.25, 0.5, 0.75, 1.0]):
    plot_vector_field(model_fm, t_val, ax)
plt.suptitle("Campo vectorial aprendido en distintos tiempos", y=1.05)
plt.tight_layout(); plt.show()


## 5.3 Animación de las trayectorias

Aquí es donde se ve la belleza: 300 partículas arrancan en $\mathcal{N}(0,I)$ y son arrastradas por el campo aprendido. Compara con el **anticipo** de la sección 1: el modelo aprendió a hacer ese transporte **sin saber el destino de cada partícula**.


In [ ]:
# Animación de las trayectorias de Flow Matching
traj_arr = torch.stack(traj_fm, dim=0).numpy()  # (T+1, N, 2)
N_show = 300
T_frames = traj_arr.shape[0]

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(real[:,0], real[:,1], s=3, alpha=0.15, color="gray")
scat = ax.scatter(traj_arr[0, :N_show, 0], traj_arr[0, :N_show, 1],
                  s=10, alpha=0.8,
                  c=np.arctan2(traj_arr[-1, :N_show, 1], traj_arr[-1, :N_show, 0]),
                  cmap="hsv")
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
title = ax.set_title("t = 0.00")

def update(i):
    scat.set_offsets(traj_arr[i, :N_show])
    title.set_text(f"t = {i/(T_frames-1):.2f}")
    return scat, title

anim_fm = animation.FuncAnimation(fig, update, frames=T_frames, interval=60, blit=False)
plt.close()
HTML(anim_fm.to_jshtml())


## 5.4 Combinado: campo vectorial + partículas

Esta es la *piezza centrale* del taller: campo vectorial y partículas, animados al mismo tiempo. Las flechas son el "agua del río", los puntos las "hojas que flotan".


In [ ]:
# Combinación: campo vectorial + partículas animadas
@torch.no_grad()
def field_grid(model, t_val, lim=3.0, grid=15):
    xs = torch.linspace(-lim, lim, grid); ys = torch.linspace(-lim, lim, grid)
    Xg, Yg = torch.meshgrid(xs, ys, indexing="xy")
    pts = torch.stack([Xg.flatten(), Yg.flatten()], dim=1).to(device)
    t = torch.full((pts.size(0),), t_val, device=device)
    v = model(pts, t).cpu().numpy()
    return Xg.numpy(), Yg.numpy(), v[:,0].reshape(grid,grid), v[:,1].reshape(grid,grid)

T_frames = 40  # menos frames para que sea más liviano
t_values = np.linspace(0, 1, T_frames)
# Pre-computamos el campo en cada tiempo
field_frames = [field_grid(model_fm, float(t), grid=14) for t in t_values]

# Trayectorias en los mismos T_frames
@torch.no_grad()
def euler_subsample(model, x0, t_values):
    traj = [x0.cpu().clone()]
    x = x0.clone()
    for i in range(1, len(t_values)):
        dt = t_values[i] - t_values[i-1]
        t = torch.full((x.size(0),), t_values[i-1], device=device)
        x = x + model(x, t) * dt
        traj.append(x.cpu().clone())
    return torch.stack(traj, 0)

torch.manual_seed(2)
x_init = torch.randn(150, 2, device=device)
traj_combo = euler_subsample(model_fm, x_init, t_values).numpy()

fig, ax = plt.subplots(figsize=(6.5, 6.5))
Xg, Yg, U, V_ = field_frames[0]
q = ax.quiver(Xg, Yg, U, V_, color="steelblue", scale=40, alpha=0.5)
scat = ax.scatter(traj_combo[0,:,0], traj_combo[0,:,1], s=15, color="crimson", edgecolor="black", linewidth=0.3, zorder=5)
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
title = ax.set_title("Campo vectorial + partículas, t=0.00")

def update(i):
    Xg, Yg, U, V_ = field_frames[i]
    q.set_UVC(U, V_)
    scat.set_offsets(traj_combo[i])
    title.set_text(f"Campo vectorial + partículas, t={t_values[i]:.2f}")
    return q, scat, title

anim_combo = animation.FuncAnimation(fig, update, frames=T_frames, interval=100, blit=False)
plt.close()
HTML(anim_combo.to_jshtml())


**Analogía física directa:** lo que ves es exactamente el movimiento de partículas en un fluido. El campo vectorial es el campo de velocidades, las trayectorias son las "líneas de corriente", y la distribución de partículas en cualquier instante satisface la ecuación de continuidad. La diferencia con un fluido real es que estamos en **muchas dimensiones** (aquí 2 solo para poder visualizar) y que el campo lo aprendimos con SGD.


---
# 6. Score function y la formulación SDE (Diffusion)

Ahora el otro camino histórico: **Diffusion Models**. Misma idea (transformar ruido en datos) pero con dos ingredientes extra: una **ecuación diferencial estocástica** (SDE) y la **score function**.

## 6.1 El proceso forward: añadir ruido progresivamente

Definimos un proceso que **destruye** los datos añadiéndoles ruido Gaussiano:

$$
x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon, \quad \varepsilon\sim\mathcal{N}(0,I),
$$

donde $\bar\alpha_t$ decrece de 1 a 0. En el límite continuo es una **SDE**:

$$
\boxed{\;d x = f(x,t)\, dt + g(t)\, dW_t\;}
$$

con $dW_t$ ruido Browniano. **Las "patadas aleatorias" del término estocástico es lo que distingue una SDE de una ODE.**


## 6.2 Visualicemos: el proceso forward en 2D

Tomamos las dos lunas y les añadimos ruido progresivamente. Mira cómo la distribución de los datos se "derrite" en $\mathcal{N}(0,I)$.


In [ ]:
# Animación del proceso forward: data -> noise
T = 200
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

x_data = sample_moons(2000)
torch.manual_seed(0)
eps = torch.randn_like(x_data)

T_frames = 40
t_indices = np.linspace(0, T-1, T_frames).astype(int)
positions_fwd = []
for ti in t_indices:
    ab = alpha_bars[ti].item()
    xt = math.sqrt(ab) * x_data + math.sqrt(1 - ab) * eps
    positions_fwd.append(xt.numpy())

fig, ax = plt.subplots(figsize=(5.5, 5.5))
scat = ax.scatter(positions_fwd[0][:,0], positions_fwd[0][:,1], s=4, alpha=0.6,
                  c=np.arctan2(x_data[:,1], x_data[:,0]), cmap="hsv")
ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
title = ax.set_title("Forward: t=0  (datos)")

def update(i):
    scat.set_offsets(positions_fwd[i])
    ti = t_indices[i]
    title.set_text(f"Forward: t={ti}/{T-1}  ($\\bar\\alpha_t$={alpha_bars[ti]:.3f})")
    return scat, title

anim_fwd = animation.FuncAnimation(fig, update, frames=T_frames, interval=80, blit=False)
plt.close()
HTML(anim_fwd.to_jshtml())


**Observa**: arrancamos con la estructura de las dos lunas (los colores te ayudan a seguir qué punto va a dónde) y terminamos con una nube isotrópica indistinguible de $\mathcal{N}(0,I)$.

## 6.3 La score function

La **score function** de una distribución $p$ es:

$$
\boxed{\; s(x) = \nabla_x \log p(x) \;}
$$

> **Analogía física:** si $-\log p(x)$ es una **energía potencial**, entonces $\nabla \log p(x)$ es la **fuerza** que empuja a una partícula hacia regiones de alta densidad — como un campo gravitacional generado por los datos.

## 6.4 Reverse-time SDE (Anderson 1982)

El milagro es que existe una **SDE reversa** que deshace el forward, ¡y solo necesita aprender la score function!

$$
\boxed{\; d x = \big[\,f(x,t) - g(t)^2 \nabla_x \log p_t(x)\,\big]\, dt + g(t)\, d\bar W_t \;}
$$

Si aprendemos $s_\theta(x,t)\approx \nabla_x \log p_t(x)$, podemos integrar de $t=1$ a $t=0$ y generar muestras.

## 6.5 Denoising Score Matching

¿Cómo aprender la score? Truco (Vincent 2011, Song & Ermon 2019): para $x_t = \alpha_t x_0 + \sigma_t \varepsilon$ la score condicional tiene forma cerrada:

$$
\nabla_{x_t} \log q(x_t|x_0) = -\frac{\varepsilon}{\sigma_t}.
$$

Entrenamos el modelo a **predecir el ruido $\varepsilon$**, que es proporcional a la score:

$$
\boxed{\; \mathcal{L}_{\text{DDPM}} = \mathbb{E}_{t,\, x_0,\, \varepsilon}\;\big\|\,\varepsilon_\theta(x_t, t) - \varepsilon\,\big\|^2 \;}
$$

## 6.6 Flow Matching vs Diffusion: dos caras de la misma moneda

| Aspecto | Flow Matching | Diffusion (Score-based) |
|---------|---------------|-------------------------|
| Ecuación | ODE: $\frac{dx}{dt}=v_t(x)$ | SDE: $dx = f\,dt + g\,dW$ |
| Aprende | Velocidad $v_\theta$ | Score $s_\theta$ (≡ predicción de ruido) |
| Sampler | Euler / RK | Euler-Maruyama, DDIM, DPM-Solver |
| Determinista | Sí | No (en SDE; sí en probability-flow ODE) |
| Camino típico | Recta $(1-t)x_0 + t x_1$ | Curvo (Gaussian schedule) |

> **Insight clave:** *cualquier* probability path puede describirse tanto por una ODE como por una SDE. Son representaciones equivalentes del mismo flujo de probabilidad. **Stable Diffusion** entrena con score matching y muestrea con ODE (DDIM, DPM-Solver) — usa lo mejor de los dos mundos.


---
# 7. Diffusion / Score Matching en 2D

Implementemos DDPM sobre el mismo dataset y verifiquemos que **otra parametrización del mismo problema** funciona igual de bien.


In [ ]:
# Schedule de ruido (DDPM con beta linear)
T = 200
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
sqrt_ab = torch.sqrt(alpha_bars).to(device)
sqrt_1mab = torch.sqrt(1 - alpha_bars).to(device)
betas = betas.to(device); alphas = alphas.to(device); alpha_bars = alpha_bars.to(device)

# Mismo TimeMLP, pero ahora predice epsilon
model_sm = TimeMLP(dim=2).to(device)
opt = torch.optim.Adam(model_sm.parameters(), lr=2e-3)

n_steps = 5000; batch_size = 512
losses_sm = []
for step in range(n_steps):
    x0 = data_sampler(batch_size).to(device)
    t = torch.randint(0, T, (batch_size,), device=device)
    eps = torch.randn_like(x0)
    xt = sqrt_ab[t][:,None]*x0 + sqrt_1mab[t][:,None]*eps
    eps_pred = model_sm(xt, t.float()/T)
    loss = F.mse_loss(eps_pred, eps)
    opt.zero_grad(); loss.backward(); opt.step()
    losses_sm.append(loss.item())
    if (step+1) % 1000 == 0:
        print(f"step {step+1:>5d}  loss = {loss.item():.4f}")


## 7.1 Animación del denoising en 2D

Mira cómo las partículas (que arrancan en $\mathcal{N}(0,I)$ ahora) son **denoised** paso a paso hasta caer en las dos lunas. Compara con la animación forward de la sección 6.2 — es exactamente la trayectoria inversa.


In [ ]:
@torch.no_grad()
def ddpm_sample_with_traj(model, n=500):
    x = torch.randn(n, 2, device=device)
    traj = [x.cpu().clone()]
    for t in reversed(range(T)):
        t_b = torch.full((n,), t, device=device, dtype=torch.long)
        eps = model(x, t_b.float()/T)
        a = alphas[t]; ab = alpha_bars[t]; b = betas[t]
        coef = (1 - a) / torch.sqrt(1 - ab)
        mean = (x - coef * eps) / torch.sqrt(a)
        if t > 0:
            x = mean + torch.sqrt(b) * torch.randn_like(x)
        else:
            x = mean
        traj.append(x.cpu().clone())
    return torch.stack(traj, dim=0)  # (T+1, N, 2)

traj_ddpm = ddpm_sample_with_traj(model_sm, n=500)
T_frames = 40
idx = np.linspace(0, traj_ddpm.shape[0]-1, T_frames).astype(int)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(real[:,0], real[:,1], s=3, alpha=0.15, color="gray")
scat = ax.scatter(traj_ddpm[idx[0],:,0], traj_ddpm[idx[0],:,1], s=10, alpha=0.7, color="C2")
ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
title = ax.set_title("Reverse SDE (DDPM)  t = T")

def update(i):
    scat.set_offsets(traj_ddpm[idx[i]].numpy())
    title.set_text(f"Reverse SDE (DDPM)  step {idx[i]}/{traj_ddpm.shape[0]-1}")
    return scat, title

anim_rev = animation.FuncAnimation(fig, update, frames=T_frames, interval=80, blit=False)
plt.close()
HTML(anim_rev.to_jshtml())


## 7.2 Comparación final: Flow Matching vs Diffusion lado a lado

Ahora el momento de la verdad — los dos modelos entrenados en el mismo dataset, lado a lado.  
- **Izquierda (FM, ODE)**: trayectorias casi rectas, deterministas.  
- **Derecha (DDPM, SDE)**: trayectorias más curvas, con "wobble" estocástico.


In [ ]:
# Animación side-by-side: FM (izquierda) vs DDPM (derecha)
T_frames = 50
# FM trajectory (uniform t from 0 to 1, n_steps=T_frames)
@torch.no_grad()
def fm_traj_n(model, n=300, n_steps=T_frames):
    x = torch.randn(n, 2, device=device)
    traj = [x.cpu().clone()]
    dt = 1.0 / n_steps
    for k in range(n_steps):
        t = torch.full((n,), k*dt, device=device)
        x = x + model(x, t) * dt
        traj.append(x.cpu().clone())
    return torch.stack(traj, dim=0)

torch.manual_seed(0)
traj_fm_side = fm_traj_n(model_fm, n=300, n_steps=T_frames)
# DDPM: subsample existing trajectory
idx_ddpm = np.linspace(0, traj_ddpm.shape[0]-1, T_frames+1).astype(int)
traj_ddpm_side = traj_ddpm[idx_ddpm, :300]

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax in axes:
    ax.scatter(real[:,0], real[:,1], s=3, alpha=0.12, color="gray")
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])

scat1 = axes[0].scatter(traj_fm_side[0,:,0], traj_fm_side[0,:,1], s=10, color="C0", alpha=0.7)
scat2 = axes[1].scatter(traj_ddpm_side[0,:,0], traj_ddpm_side[0,:,1], s=10, color="C2", alpha=0.7)
axes[0].set_title("Flow Matching (ODE)")
axes[1].set_title("Diffusion (DDPM, SDE)")
sup = fig.suptitle("Comparación de trayectorias en el mismo dataset", y=0.98)

def update(i):
    scat1.set_offsets(traj_fm_side[i].numpy())
    scat2.set_offsets(traj_ddpm_side[i].numpy())
    return scat1, scat2

anim_cmp = animation.FuncAnimation(fig, update, frames=T_frames+1, interval=80, blit=False)
plt.close()
HTML(anim_cmp.to_jshtml())


## 7.3 Score field aprendido

Igual que con FM visualizamos $v_\theta$, ahora visualizamos la score $s(x,t)=-\varepsilon_\theta(x,t)/\sigma_t$ — apunta hacia regiones de mayor densidad.


In [ ]:
@torch.no_grad()
def plot_score_field(model, t_idx, ax, lim=3.0, grid=18):
    xs = torch.linspace(-lim, lim, grid); ys = torch.linspace(-lim, lim, grid)
    Xg, Yg = torch.meshgrid(xs, ys, indexing="xy")
    pts = torch.stack([Xg.flatten(), Yg.flatten()], dim=1).to(device)
    eps = model(pts, torch.full((pts.size(0),), t_idx/T, device=device))
    score = -eps / sqrt_1mab[t_idx]
    s = score.cpu().numpy()
    ax.quiver(Xg.numpy(), Yg.numpy(), s[:,0].reshape(grid,grid), s[:,1].reshape(grid,grid),
              scale=80, alpha=0.7, color="C2")
    ax.set_title(f"$s_\\theta(x, t={t_idx})$"); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 5, figsize=(15,3))
for ax, ti in zip(axes, [T-1, 3*T//4, T//2, T//4, 1]):
    plot_score_field(model_sm, ti, ax)
plt.suptitle("Score function aprendida (cerca de los datos, las flechas apuntan a las modas)", y=1.05)
plt.tight_layout(); plt.show()


---
# 8. Arquitecturas: U-Net y time embeddings

Hasta ahora usamos un MLP para datos 2D. Para imágenes la arquitectura de elección es la **U-Net**.

## 8.1 Estructura de la U-Net

Tres propiedades clave:
1. **Multi-escala**: downsampling + upsampling permite razonar sobre estructura global y detalle fino.
2. **Skip connections**: el detalle de alta frecuencia se preserva (esencial para predecir bien el ruido).
3. **Mismo tamaño in/out**: input $x_t \in \mathbb{R}^{C\times H\times W}$, output $\varepsilon_\theta \in \mathbb{R}^{C\times H\times W}$.


In [ ]:
# Diagrama de U-Net dibujado con matplotlib
def draw_box(ax, x, y, w, h, label, color="#a8d5e5", lw=1.5):
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05,rounding_size=0.1",
                         facecolor=color, edgecolor="black", linewidth=lw)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=9)

def arrow(ax, x1, y1, x2, y2, color="black", style="->", curve=0):
    arr = FancyArrowPatch((x1, y1), (x2, y2),
                          arrowstyle=style, mutation_scale=15,
                          connectionstyle=f"arc3,rad={curve}", color=color, lw=1.5)
    ax.add_patch(arr)

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14); ax.set_ylim(0, 7); ax.axis("off")

# Encoder (downsampling)
draw_box(ax, 0.5, 4.5, 1.6, 1.2, "Conv block\n32x32, C=32", color="#a8d5e5")
draw_box(ax, 2.6, 4.0, 1.5, 1.0, "Down\n16x16, C=64", color="#7fbfd4")
draw_box(ax, 4.5, 3.5, 1.3, 0.8, "Down\n8x8, C=128", color="#56a9c3")

# Bottleneck
draw_box(ax, 6.2, 3.0, 1.4, 0.7, "Bottleneck\n+ Attention", color="#ff9999")

# Decoder (upsampling)
draw_box(ax, 8.0, 3.5, 1.3, 0.8, "Up\n8x8, C=128", color="#56a9c3")
draw_box(ax, 9.7, 4.0, 1.5, 1.0, "Up\n16x16, C=64", color="#7fbfd4")
draw_box(ax, 11.7, 4.5, 1.6, 1.2, "Conv block\n32x32, C=32", color="#a8d5e5")

# Flechas de flujo
arrow(ax, 2.1, 5.1, 2.6, 4.5)
arrow(ax, 4.1, 4.5, 4.5, 3.9)
arrow(ax, 5.8, 3.9, 6.2, 3.35)
arrow(ax, 7.6, 3.35, 8.0, 3.9)
arrow(ax, 9.3, 3.9, 9.7, 4.5)
arrow(ax, 11.2, 5.0, 11.7, 5.1)

# Skip connections (curvas arriba)
arrow(ax, 1.3, 5.7, 12.5, 5.7, color="#cc3300", style="->", curve=-0.25)
arrow(ax, 3.35, 5.0, 10.45, 5.0, color="#cc3300", style="->", curve=-0.3)
arrow(ax, 5.15, 4.3, 8.65, 4.3, color="#cc3300", style="->", curve=-0.3)

ax.text(7, 6.55, "Skip connections (preservan detalle fino)", ha="center", color="#cc3300", fontsize=10, style="italic")

# Input / Output
ax.text(0.5, 6.4, "Input:\n$x_t$  +  $t$  (+  $c$)", ha="left", fontsize=11, weight="bold")
ax.text(13.3, 6.4, "Output:\n$\\varepsilon_\\theta(x_t,t)$", ha="right", fontsize=11, weight="bold")
arrow(ax, 0.0, 5.7, 0.5, 5.7, color="black")
arrow(ax, 13.3, 5.7, 13.8, 5.7, color="black")

# Time embedding (abajo)
draw_box(ax, 5.5, 0.7, 3, 1.0, "Time embedding\n(sinusoidal → MLP)", color="#ffe699")
arrow(ax, 7, 1.7, 7, 2.95, color="#888", style="->")
ax.text(7.15, 2.35, "$t$ se inyecta en cada bloque", fontsize=9, style="italic", color="#555")

# Etiquetas
ax.text(2.85, 2.5, "Encoder\n(downsampling)", ha="center", fontsize=10, weight="bold", color="#0066aa")
ax.text(11, 2.5, "Decoder\n(upsampling)", ha="center", fontsize=10, weight="bold", color="#0066aa")

plt.title("Arquitectura U-Net para diffusion (32×32, 3 escalas)", fontsize=12, pad=10)
plt.show()


## 8.2 Time embedding sinusoidal

El modelo necesita "saber" en qué $t$ está operando. Igual que en Transformers, codificamos $t$ con funciones sinusoidales de múltiples frecuencias:

$$
\text{embed}(t) = \big[\sin(\omega_0 t), \cos(\omega_0 t), \sin(\omega_1 t), \cos(\omega_1 t), \ldots\big]
$$

Cada bloque del U-Net recibe ese embedding (proyectado a la dimensión adecuada) y lo suma a sus features.

**¿Por qué no simplemente one-hot?** Si usáramos un vector one-hot de tamaño $T$ (e.g. 1000):
- La dimensionalidad sería enorme solo para codificar un escalar.
- Timesteps vecinos ($t=500$ y $t=501$) serían ortogonales — tan "diferentes" como $t=500$ y $t=999$. No hay noción de cercanía.
- En Flow Matching, $t \in [0,1]$ es **continuo** ($t = 0.3718...$), así que un one-hot discreto ni siquiera aplica.

El embedding sinusoidal resuelve todo esto: es compacto (32 dims bastan), **continuo** (acepta cualquier $t$ real), y preserva cercanía ($t$ similares → vectores similares).

> **Analogía del reloj:** piensa en las manecillas de un reloj — la de las horas gira lento (frecuencia baja) y la de los minutos gira rápido (frecuencia alta). Con ambas puedes identificar de forma única cualquier momento del día. El embedding sinusoidal hace exactamente eso: usa múltiples "manecillas" (frecuencias) para dar una representación rica y única de cada instante $t$.

In [ ]:
# Visualización del time embedding
def sinusoidal_emb(t, dim=32):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half) / half)
    args = t[:, None] * freqs[None] * 2 * math.pi
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

ts = torch.linspace(0, 1, 200)
emb = sinusoidal_emb(ts, dim=32).numpy()

fig, ax = plt.subplots(figsize=(10, 3.5))
im = ax.imshow(emb.T, aspect="auto", cmap="RdBu_r", interpolation="nearest")
ax.set_xlabel("t (paso de tiempo, normalizado)")
ax.set_ylabel("dimensión del embedding")
ax.set_title("Time embedding sinusoidal: cada t produce un vector único de 32 dimensiones")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.show()


---
# 9. Diffusion sobre imágenes — MNIST

Vamos a entrenar un DDPM compacto sobre **MNIST** ($28\times 28$ en escala de grises). En GPU Colab, ~5-8 minutos de entrenamiento son suficientes para generar dígitos legibles.


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

tfm = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
mnist = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
loader = DataLoader(mnist, batch_size=128, shuffle=True, num_workers=2, drop_last=True)
print(f"Tamaño del dataset: {len(mnist)}")


In [ ]:
# U-Net pequeño usando diffusers
from diffusers import UNet2DModel, DDPMScheduler

unet_mnist = UNet2DModel(
    sample_size=32,
    in_channels=1, out_channels=1,
    layers_per_block=2,
    block_out_channels=(32, 64, 128),
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
).to(device)
print(f"Parámetros: {sum(p.numel() for p in unet_mnist.parameters())/1e6:.2f}M")

scheduler_mnist = DDPMScheduler(num_train_timesteps=1000, beta_schedule="linear")


In [ ]:
# Entrenamiento
opt = torch.optim.AdamW(unet_mnist.parameters(), lr=1e-4)
n_epochs = 8   # subir a 10-15 si dispones de más tiempo

unet_mnist.train()
losses_mnist = []
for epoch in range(n_epochs):
    for step, (x, _) in enumerate(loader):
        x = x.to(device)
        noise = torch.randn_like(x)
        t = torch.randint(0, scheduler_mnist.config.num_train_timesteps,
                          (x.size(0),), device=device).long()
        x_noisy = scheduler_mnist.add_noise(x, noise, t)
        pred = unet_mnist(x_noisy, t).sample
        loss = F.mse_loss(pred, noise)
        opt.zero_grad(); loss.backward(); opt.step()
        losses_mnist.append(loss.item())
        if step % 100 == 0:
            print(f"epoch {epoch+1}/{n_epochs}  step {step:4d}  loss {loss.item():.4f}")

plt.figure(figsize=(6,2.5))
plt.plot(losses_mnist); plt.title("Loss MNIST"); plt.xlabel("step"); plt.show()

In [ ]:
@torch.no_grad()
def sample_mnist(unet, scheduler, n=16, n_steps=200, save_traj=False):
    scheduler.set_timesteps(n_steps)
    x = torch.randn(n, 1, 32, 32, device=device)
    traj = [x.cpu().clone()] if save_traj else None
    for t in scheduler.timesteps:
        pred = unet(x, t).sample
        x = scheduler.step(pred, t, x).prev_sample
        if save_traj: traj.append(x.cpu().clone())
    return (x.cpu(), traj) if save_traj else x.cpu()

unet_mnist.eval()
samples_mnist, traj_mnist = sample_mnist(unet_mnist, scheduler_mnist, n=8, n_steps=200, save_traj=True)

# Final samples
fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for ax, img in zip(axes, samples_mnist):
    img = (img.clamp(-1,1)+1)/2
    ax.imshow(img.squeeze(), cmap="gray"); ax.axis("off")
plt.suptitle("Dígitos generados (DDPM no condicional)", y=1.1); plt.show()


## 9.1 Animación: los dígitos emergiendo del ruido

Cada partícula es una "imagen" en $\mathbb{R}^{1024}$ que arranca como ruido y va siendo transportada por el campo de score hasta caer en la variedad de los dígitos.


In [ ]:
# Animación del denoising MNIST
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
ims = []
for j, ax in enumerate(axes):
    ax.axis("off")
    im = ax.imshow(traj_mnist[0][j, 0].clamp(-1,1).numpy(), cmap="gray", vmin=-1, vmax=1)
    ims.append(im)
title = fig.suptitle("step 0 / 200 — ruido puro", fontsize=12, y=1.05)

T_frames = 40
idx = np.linspace(0, len(traj_mnist)-1, T_frames).astype(int)
def update(i):
    for j, im in enumerate(ims):
        im.set_data(traj_mnist[idx[i]][j, 0].clamp(-1,1).numpy())
    title.set_text(f"step {idx[i]} / {len(traj_mnist)-1}")
    return ims + [title]

anim_mnist = animation.FuncAnimation(fig, update, frames=T_frames, interval=100, blit=False)
plt.close()
HTML(anim_mnist.to_jshtml())


*Recuerda la analogía:* cada uno de estos 8 dígitos es una **partícula** que arrancó en $\mathbb{R}^{1024}$ como ruido y fue arrastrada por el campo (la score) durante 200 pasos hasta caer en la variedad de los dígitos. Es exactamente lo mismo que vimos en 2D, pero en 1024 dimensiones — la matemática es invariante a la dimensión.


---
# 10. Classifier-Free Guidance (CFG)

Hasta aquí muestreamos **incondicionalmente**: el modelo no sabe qué generar. En la práctica queremos pedirle "dame un 7" o "un gato sentado en una bicicleta". Necesitamos condicionar.

## 10.1 Modelo condicional + el truco de CFG (Ho & Salimans 2022)

Pasamos al modelo una **etiqueta de clase** $c$:
$$
\varepsilon_\theta(x_t, t, c).
$$

Durante el entrenamiento, con probabilidad $p_{\text{drop}}$ (≈0.1) **reemplazamos $c$ por un token nulo $\varnothing$**. Así el mismo modelo aprende ambos: condicional e incondicional.

En **inferencia** combinamos las dos predicciones con una **escala de guidance** $w \ge 0$:

$$
\boxed{\;\tilde\varepsilon(x,t,c) = (1+w)\,\varepsilon_\theta(x,t,c) - w\,\varepsilon_\theta(x,t,\varnothing)\;}
$$

- $w=0$: muestreo condicional puro.
- $w>0$: amplifica la dirección "hacia la clase $c$" → muestras más fieles pero menos diversas.

> **Intuición física:** la guidance es un **campo de fuerza extra** que estiramos en la dirección "hacia la clase $c$" para empujar las partículas hacia esa moda.


In [ ]:
# U-Net condicional con class embedding
class CondUNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.unet = UNet2DModel(
            sample_size=32, in_channels=1, out_channels=1,
            layers_per_block=2, block_out_channels=(32, 64, 128),
            down_block_types=("DownBlock2D", "AttnDownBlock2D", "DownBlock2D"),
            up_block_types=("UpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
            num_class_embeds=num_classes + 1,   # +1 = token nulo
        )
        self.null_idx = num_classes

    def forward(self, x, t, c=None):
        if c is None:
            c = torch.full((x.size(0),), self.null_idx, device=x.device, dtype=torch.long)
        return self.unet(x, t, class_labels=c).sample

unet_cond = CondUNet().to(device)
print(f"Parámetros: {sum(p.numel() for p in unet_cond.parameters())/1e6:.2f}M")


In [ ]:
opt = torch.optim.AdamW(unet_cond.parameters(), lr=1e-4)
scheduler_cond = DDPMScheduler(num_train_timesteps=1000, beta_schedule="linear")
P_DROP = 0.1
n_epochs = 3

unet_cond.train()
for epoch in range(n_epochs):
    for step, (x, y) in enumerate(loader):
        x = x.to(device); y = y.to(device)
        mask = torch.rand(y.shape, device=device) < P_DROP
        y_in = torch.where(mask, torch.full_like(y, unet_cond.null_idx), y)
        noise = torch.randn_like(x)
        t = torch.randint(0, 1000, (x.size(0),), device=device).long()
        x_noisy = scheduler_cond.add_noise(x, noise, t)
        pred = unet_cond(x_noisy, t, y_in)
        loss = F.mse_loss(pred, noise)
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 100 == 0:
            print(f"epoch {epoch+1}/{n_epochs}  step {step:4d}  loss {loss.item():.4f}")


In [ ]:
@torch.no_grad()
def sample_cond(unet, scheduler, classes, n_steps=150, w=3.0):
    scheduler.set_timesteps(n_steps)
    classes = torch.tensor(classes, device=device, dtype=torch.long)
    n = classes.size(0)
    null = torch.full((n,), unet.null_idx, device=device, dtype=torch.long)
    x = torch.randn(n, 1, 32, 32, device=device)
    for t in scheduler.timesteps:
        eps_c   = unet(x, t, classes)
        eps_uc  = unet(x, t, null)
        eps = (1 + w) * eps_c - w * eps_uc
        x = scheduler.step(eps, t, x).prev_sample
    return x.cpu()

unet_cond.eval()
fig, axes = plt.subplots(4, 10, figsize=(14, 5.5))
for row, w in enumerate([0.0, 1.0, 3.0, 7.0]):
    samples_c = sample_cond(unet_cond, scheduler_cond, list(range(10)), n_steps=150, w=w)
    for col in range(10):
        img = (samples_c[col].clamp(-1,1)+1)/2
        axes[row, col].imshow(img.squeeze(), cmap="gray"); axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(f"w={w}", rotation=0, ha="right", va="center")
        if row == 0:
            axes[row, col].set_title(str(col))
plt.suptitle("Classifier-Free Guidance: efecto de $w$ (escala de guidance)", y=1.02)
plt.tight_layout(); plt.show()


**Análisis:** a medida que aumenta $w$, los dígitos son **más fieles a la clase pedida** pero **menos diversos**; si $w$ es muy grande aparecen artefactos. Stable Diffusion usa típicamente $w\in[5, 12]$.


---
# 11. Latent Diffusion — *de MNIST a Stable Diffusion*

## 11.1 El problema de escalar a imágenes grandes

Para generar imágenes de $512\times 512$ a color, $x \in \mathbb{R}^{786432}$. Hacer diffusion ahí directamente requiere U-Nets gigantes y muchos pasos de muestreo en alta resolución.

## 11.2 La idea (Rombach et al. 2022)

> **Hagamos el diffusion en un espacio latente comprimido.**

1. Pre-entrenar un **autoencoder** (típicamente VAE):
   - Encoder: $\mathcal{E}: \mathbb{R}^{3\times H\times W} \to \mathbb{R}^{4\times H/8\times W/8}$
   - Decoder: $\mathcal{D}$ que reconstruye.
2. Entrenar el U-Net de diffusion **en el espacio latente** $z = \mathcal{E}(x)$.
3. Para generar: muestrear $z \sim p_\theta$ con el U-Net, decodificar $x = \mathcal{D}(z)$.

Esto reduce la dimensionalidad ~48× y es la base de **Stable Diffusion**, **SDXL**, **Flux**, etc.


In [ ]:
# Diagrama del pipeline de Latent Diffusion
fig, ax = plt.subplots(figsize=(15, 5.5))
ax.set_xlim(0, 15); ax.set_ylim(0, 5.5); ax.axis("off")

# Texto -> Text Encoder
draw_box(ax, 0.2, 4.0, 2.0, 1.0, "Texto:\n\"a fox in a forest\"", color="#e6f2ff")
draw_box(ax, 0.2, 2.6, 2.0, 1.0, "Text Encoder\n(CLIP)", color="#b3d4ff")
arrow(ax, 1.2, 4.0, 1.2, 3.6)
ax.text(1.2, 2.3, "$c$ (embedding)", ha="center", fontsize=9, style="italic")

# Ruido inicial
draw_box(ax, 2.8, 0.4, 1.5, 1.0, "$z_T \\sim \\mathcal{N}(0,I)$\nlatent (4×64×64)", color="#dddddd")

# U-Net loop
draw_box(ax, 4.8, 2.0, 4.5, 1.7, "U-Net (latent)\n$\\varepsilon_\\theta(z_t, t, c)$  +  CFG", color="#ffe6cc", lw=2)
# Bucle de denoising (flecha curva)
arrow(ax, 9.3, 2.0, 9.3, 1.4, color="#cc6600")
arrow(ax, 9.3, 1.4, 4.8, 1.4, color="#cc6600", style="->", curve=0.0)
arrow(ax, 4.8, 1.4, 4.8, 2.0, color="#cc6600")
ax.text(7.05, 1.0, "iterar $T$ pasos (Euler / DDIM / DPM-Solver)", ha="center", fontsize=9, style="italic", color="#cc6600")

# Entradas al U-Net
arrow(ax, 4.3, 0.9, 4.8, 2.2)
arrow(ax, 2.3, 3.1, 4.8, 3.1)
ax.text(2.55, 3.3, "$c$", fontsize=11, color="#0066cc")
ax.text(3.6, 1.8, "$z_T$", fontsize=11)

# z_0 output
draw_box(ax, 9.8, 2.4, 1.5, 0.9, "$z_0$\n(4×64×64)", color="#cce6cc")
arrow(ax, 9.3, 2.85, 9.8, 2.85)

# VAE Decoder
draw_box(ax, 11.7, 2.4, 1.5, 0.9, "VAE\nDecoder $\\mathcal{D}$", color="#99cc99")
arrow(ax, 11.3, 2.85, 11.7, 2.85)

# Output image
draw_box(ax, 13.4, 1.8, 1.4, 2.0, "Imagen\n3×512×512", color="#ffcccc")
arrow(ax, 13.2, 2.85, 13.4, 2.85)

# Labels
ax.text(7.05, 4.6, "Loop de denoising en el espacio latente (×48 más rápido que en píxeles)",
        ha="center", fontsize=12, weight="bold", color="#cc6600")
ax.text(12.45, 4.0, "Decodificación final\n(latent → píxeles)",
        ha="center", fontsize=10, style="italic")

plt.title("Pipeline de Latent Diffusion (Stable Diffusion)", fontsize=13, pad=10)
plt.show()


## 11.3 Demo: muestreo con Stable Diffusion

Cargamos un modelo de Stable Diffusion **destilado y compacto** (`nota-ai/bk-sdm-tiny`, ~700 MB) y le pedimos una imagen. La inferencia toma pocos segundos en Colab con GPU.


In [ ]:
from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained(
    "nota-ai/bk-sdm-tiny",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,
).to(device)

prompt = "a watercolor painting of a fox in a forest, soft lighting"
image = pipe(prompt, num_inference_steps=25, guidance_scale=7.5).images[0]
plt.figure(figsize=(5,5)); plt.imshow(image); plt.axis("off"); plt.title(prompt); plt.show()


## 11.4 Lo que está pasando bajo el capó

Dentro de `pipe(...)` sucede **exactamente** lo que hemos visto:

1. **Texto → embedding** con CLIP.
2. **$z_T \sim \mathcal{N}(0, I)$** en $\mathbb{R}^{4\times 64\times 64}$ (latente, no píxeles).
3. **Loop de denoising** en el latente:
   - U-Net predice $\varepsilon_\theta(z_t, t, c)$.
   - **CFG** combina con la versión incondicional.
   - Solver (DDIM, DPM-Solver) avanza.
4. **VAE decoder** convierte $z_0$ → imagen.

Todo lo que aprendiste en 2D y MNIST escala — la matemática es la misma.


---
# Resumen final

## La gran unificación

Flow Matching y Diffusion son **dos formulaciones equivalentes** del mismo problema: aprender un **transporte** entre $\mathcal{N}(0,I)$ y $p_{\text{data}}$.

| Concepto | Flow Matching | Diffusion |
|----------|---------------|-----------|
| Camino | Recto, $x_t=(1-t)x_0+tx_1$ | Curvo (Gaussian schedule) |
| Modelo predice | Velocidad $v_\theta$ | Ruido $\varepsilon_\theta$ (≡ score) |
| Dinámica | ODE determinista | SDE estocástica (o ODE equivalente) |
| Sampler | Euler, RK | Euler-Maruyama, DDIM, DPM-Solver |
| Loss | $\|v_\theta - (x_1-x_0)\|^2$ | $\|\varepsilon_\theta - \varepsilon\|^2$ |

Ambos resuelven la **ecuación de continuidad** en un fluido de probabilidad de alta dimensión. Cada muestra es una **partícula** arrastrada por el campo aprendido.

## ¿Qué construiste hoy?

- Refresco de ODEs con campos vectoriales analíticos animados.
- Un modelo de Flow Matching que aprende a muestrear distribuciones 2D, con campo vectorial y trayectorias animadas.
- Un modelo de Diffusion (DDPM) sobre los mismos datos, animado.
- Comparación lado-a-lado FM vs Diffusion.
- Un U-Net para MNIST con muestreo animado.
- Modelo condicional con **classifier-free guidance**.
- Diagrama y demo de **latent diffusion** (Stable Diffusion).

## Próximos pasos

- **Rectified Flow** (Liu et al. 2023): "enderezar" trayectorias para muestrear con 1-4 pasos.
- **Consistency Models** (Song et al. 2023): muestreo en un solo paso.
- **DPM-Solver / Heun / Karras schedules**: samplers de mayor calidad/eficiencia.
- **Diffusion Transformers (DiT)** que reemplazan U-Net por Transformers.
- **Score Distillation Sampling** para 3D (DreamFusion).
- **Audio/video diffusion**: misma matemática, otra modalidad.

## Referencias clave

- Ho, Jain, Abbeel — *Denoising Diffusion Probabilistic Models* (2020).
- Song et al. — *Score-Based Generative Modeling through SDEs* (2021).
- Lipman et al. — *Flow Matching for Generative Modeling* (2023).
- Liu et al. — *Flow Straight and Fast: Rectified Flow* (2023).
- Rombach et al. — *High-Resolution Image Synthesis with Latent Diffusion Models* (2022).
- Yang Song's blog: <https://yang-song.net/blog/2021/score/>
- Lilian Weng's blog: <https://lilianweng.github.io/posts/2021-07-11-diffusion-models/>
- Hugging Face Diffusion Course: <https://huggingface.co/learn/diffusion-course>

---

> **¡Gracias por venir!** Con un único concepto unificado — *aprender un campo vectorial que transporta una distribución* — podemos generar dígitos, fotos, audio, video y casi cualquier modalidad de datos. Es matemática vieja (fluidos, transporte óptimo) hecha nueva por las redes neuronales.
